# Raw Data Viewer

Inspect cached NVD and OSV records side by side without drawing conclusions.


In [7]:
import json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
NVD_DIR = ROOT / 'data' / 'raw' / 'nvd'
OSV_DIR = ROOT / 'data' / 'raw' / 'osv'

def load_json(path):
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)

def take_records(directory, package_names, key, per_package=5):
    samples = {}
    for package_name in package_names:
        payload = load_json(directory / f'{package_name}.json')
        samples[package_name] = payload.get(key, [])[:per_package]
    return samples

nvd_samples = take_records(NVD_DIR, ['lodash', 'log4j-core', 'express'], 'vulnerabilities', per_package=5)
for package_name, records in nvd_samples.items():
    print(f'--- NVD {package_name} ({len(records)} sampled records) ---')
    for record in records:
        cve = record.get('cve', {})
        cve_id = cve.get('id')
        desc = next((d.get('value') for d in cve.get('descriptions', []) if d.get('lang') == 'en'), '')
        print(cve_id)
        print(desc)
        print()


--- NVD lodash (5 sampled records) ---
CVE-2018-3721
lodash node module before 4.17.5 suffers from a Modification of Assumed-Immutable Data (MAID) vulnerability via defaultsDeep, merge, and mergeWith functions, which allows a malicious user to modify the prototype of "Object" via __proto__, causing the addition or modification of an existing property that will exist on all objects.

CVE-2018-16487
A prototype pollution vulnerability was found in lodash <4.17.11 where the functions merge, mergeWith, and defaultsDeep can be tricked into adding or modifying properties of Object.prototype.

CVE-2019-1010266
lodash prior to 4.17.11 is affected by: CWE-400: Uncontrolled Resource Consumption. The impact is: Denial of service. The component is: Date handler. The attack vector is: Attacker provides very long strings, which the library attempts to match using a regular expression. The fixed version is: 4.17.11.

CVE-2019-10744
Versions of lodash lower than 4.17.12 are vulnerable to Prototype Pol

In [8]:
osv_samples = take_records(OSV_DIR, ['lodash', 'log4j-core', 'express'], 'vulns', per_package=5)
for package_name, records in osv_samples.items():
    print(f'--- OSV {package_name} ({len(records)} sampled records) ---')
    for record in records:
        vuln_id = record.get('id')
        summary = record.get('summary', '')
        details = record.get('details', '')
        print(vuln_id)
        print(summary)
        print(details)
        print()


--- OSV lodash (5 sampled records) ---
GHSA-29mw-wpgm-hmr9
Regular Expression Denial of Service (ReDoS) in lodash
All versions of package lodash prior to 4.17.21 are vulnerable to Regular Expression Denial of Service (ReDoS) via the `toNumber`, `trim` and `trimEnd` functions. 

Steps to reproduce (provided by reporter Liyuan Chen):
```js
var lo = require('lodash');

function build_blank(n) {
    var ret = "1"
    for (var i = 0; i < n; i++) {
        ret += " "
    }
    return ret + "1";
}
var s = build_blank(50000) var time0 = Date.now();
lo.trim(s) 
var time_cost0 = Date.now() - time0;
console.log("time_cost0: " + time_cost0);
var time1 = Date.now();
lo.toNumber(s) var time_cost1 = Date.now() - time1;
console.log("time_cost1: " + time_cost1);
var time2 = Date.now();
lo.trimEnd(s);
var time_cost2 = Date.now() - time2;
console.log("time_cost2: " + time_cost2);
```

GHSA-35jh-r3h4-6jhm
Command Injection in lodash
`lodash` versions prior to 4.17.21 are vulnerable to Command Injection vi

In [9]:
def index_nvd_by_cve(directory, package_names):
    indexed = {}
    for package_name in package_names:
        payload = load_json(directory / f'{package_name}.json')
        for record in payload.get('vulnerabilities', []):
            cve = record.get('cve', {})
            cve_id = cve.get('id')
            if cve_id:
                indexed[cve_id] = record
    return indexed

def find_osv_records_with_cve_aliases(directory, package_names):
    matches = []
    for package_name in package_names:
        payload = load_json(directory / f'{package_name}.json')
        for record in payload.get('vulns', []):
            aliases = record.get('aliases', [])
            cve_aliases = [alias for alias in aliases if isinstance(alias, str) and alias.startswith('CVE-')]
            if cve_aliases:
                matches.append((package_name, record, cve_aliases))
    return matches

nvd_index = index_nvd_by_cve(NVD_DIR, ['lodash', 'log4j-core', 'express', 'openssl', 'django', 'flask', 'axios'])
osv_alias_matches = find_osv_records_with_cve_aliases(OSV_DIR, ['lodash', 'log4j-core', 'express', 'openssl', 'django', 'flask', 'axios'])

shown = 0
for package_name, osv_record, cve_aliases in osv_alias_matches:
    for cve_id in cve_aliases:
        nvd_record = nvd_index.get(cve_id)
        if not nvd_record:
            continue
        print(f'=== {cve_id} ===')
        print('NVD:')
        print(json.dumps(nvd_record, indent=2, ensure_ascii=False))
        print('OSV:')
        print(json.dumps(osv_record, indent=2, ensure_ascii=False))
        print()
        shown += 1
        if shown >= 3:
            break
    if shown >= 3:
        break


=== CVE-2020-28500 ===
NVD:
{
  "cve": {
    "id": "CVE-2020-28500",
    "sourceIdentifier": "report@snyk.io",
    "published": "2021-02-15T11:15:12.397",
    "lastModified": "2026-06-17T03:10:32.757",
    "vulnStatus": "Modified",
    "cveTags": [],
    "descriptions": [
      {
        "lang": "en",
        "value": "Lodash versions prior to 4.17.21 are vulnerable to Regular Expression Denial of Service (ReDoS) via the toNumber, trim and trimEnd functions."
      },
      {
        "lang": "es",
        "value": "Las versiones de Lodash anteriores a la 4.17.21 son vulnerables a la denegación de servicio por expresiones regulares (ReDoS) a través de las funciones toNumber, trim y trimEnd"
      }
    ],
    "affected": [
      {
        "source": "report@snyk.io",
        "affectedData": [
          {
            "vendor": "n/a",
            "product": "Lodash",
            "versions": [
              {
                "version": "versions prior to 4.17.21",
                "status": 

In [10]:
osv_payload = load_json(OSV_DIR / 'log4j-core.json')
for record in osv_payload.get('vulns', [])[:3]:
    print(f"--- {record.get('id')} ---")
    print(json.dumps(record.get('affected', []), indent=2, ensure_ascii=False))
    print()


--- GHSA-3pxv-7cmr-fjr4 ---
[
  {
    "package": {
      "name": "org.apache.logging.log4j:log4j-core",
      "ecosystem": "Maven",
      "purl": "pkg:maven/org.apache.logging.log4j/log4j-core"
    },
    "ranges": [
      {
        "type": "ECOSYSTEM",
        "events": [
          {
            "introduced": "2.0-alpha1"
          },
          {
            "fixed": "2.25.4"
          }
        ]
      }
    ],
    "versions": [
      "2.0",
      "2.0-alpha1",
      "2.0-alpha2",
      "2.0-beta1",
      "2.0-beta2",
      "2.0-beta3",
      "2.0-beta4",
      "2.0-beta5",
      "2.0-beta6",
      "2.0-beta7",
      "2.0-beta8",
      "2.0-beta9",
      "2.0-rc1",
      "2.0-rc2",
      "2.0.1",
      "2.0.2",
      "2.1",
      "2.10.0",
      "2.11.0",
      "2.11.1",
      "2.11.2",
      "2.12.0",
      "2.12.1",
      "2.12.2",
      "2.12.3",
      "2.12.4",
      "2.13.0",
      "2.13.1",
      "2.13.2",
      "2.13.3",
      "2.14.0",
      "2.14.1",
      "2.15.0",
      "2

## Chunking decision


## Merge strategy decision


## Version-range format notes
